In [12]:
import pandas as pd

if 'roll_number' not in globals():
    roll_number = 23

last_two_digits = [int(d) for d in str(roll_number)[-2:]]

fixed_faq = [
    {
        "question": "How can I reset my student portal password?",
        "answer": "Use the 'Forgot Password' link on the student portal login page and follow the password reset email instructions.",
        "keywords": ["password", "login", "security,student"],
        "category": "account"
    },
    {
        "question": "How can I pay my tuition fees online?",
        "answer": "Go to the billing section in the student portal, select your term, and complete the payment using the secure payment gateway.",
        "keywords": ["fees", "payment", "receipt"],
        "category": "billing"
    },
    {
        "question": "Where can I check my class timetable?",
        "answer": "Your class timetable is available under the Academic Services section in the student portal or the timetable dashboard.",
        "keywords": ["timetable", "classes", "schedule"],
        "category": "general"
    },
    {
        "question": "How can I contact an academic advisor?",
        "answer": "You can contact an academic advisor through the student support office, advisor email, or the online support request form.",
        "keywords": ["advisor", "academic", "support"],
        "category": "general"
    },
    
]

custom_faq = [
    {
        "question": "Where can I find information about college holidays?",
        "answer": "You can check the academic calendar in the student portal for the complete list of college holidays.",
        "keywords": ["holidays", "calendar", "academic"],
        "category": ["billing", "account", "general"][last_two_digits[0] % 3]
    },
    {
        "question": "How can I check whether my tuition fee payment was successful?",
        "answer": "You can verify your tuition fee payment status in the billing section of the student portal and view the payment receipt.",
        "keywords": ["payment", "billing", "receipt"],
        "category": ["billing", "account", "general"][last_two_digits[1] % 3]
    }
]

all_faq = fixed_faq + custom_faq


In [13]:
df = pd.DataFrame(all_faq)

In [14]:
print(df)

                                            question  \
0        How can I reset my student portal password?   
1              How can I pay my tuition fees online?   
2              Where can I check my class timetable?   
3             How can I contact an academic advisor?   
4  Where can I find information about college hol...   
5  How can I check whether my tuition fee payment...   

                                              answer  \
0  Use the 'Forgot Password' link on the student ...   
1  Go to the billing section in the student porta...   
2  Your class timetable is available under the Ac...   
3  You can contact an academic advisor through th...   
4  You can check the academic calendar in the stu...   
5  You can verify your tuition fee payment status...   

                              keywords category  
0  [password, login, security,student]  account  
1             [fees, payment, receipt]  billing  
2       [timetable, classes, schedule]  general  
3         [adv

In [15]:
def score_hypothesis(query):
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        keywords = set(word.lower() for word in row["keywords"])

        
        matches = query_words.intersection(keywords)

        
        if len(keywords) > 0:
            confidence = len(matches) / len(keywords)
        else:
            confidence = 0

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": list(matches),
                "confidence": confidence
            })

    # Rank by confidence, highest first
    results.sort(key=lambda x: x["confidence"], reverse=True)

    return results

In [16]:
query = "How can I make a payment for my fees?"

results = score_hypothesis(query)

for result in results:
    print(result)

{'question': 'How can I pay my tuition fees online?', 'answer': 'Go to the billing section in the student portal, select your term, and complete the payment using the secure payment gateway.', 'category': 'billing', 'matched_keywords': ['payment'], 'confidence': 0.3333333333333333}
{'question': 'How can I check whether my tuition fee payment was successful?', 'answer': 'You can verify your tuition fee payment status in the billing section of the student portal and view the payment receipt.', 'category': 'billing', 'matched_keywords': ['payment'], 'confidence': 0.3333333333333333}


In [17]:
def same_category(category_name, df):
    return df[df["category"] == category_name]["question"].tolist()
category_name = "general"
result = same_category(category_name, df)
print(result)

['Where can I check my class timetable?', 'How can I contact an academic advisor?', 'Where can I find information about college holidays?']


In [18]:
# Pick the first entry
index = 0

# Ask the user for a new keyword
new_keyword = input("Enter a new keyword: ")

# Add the new keyword to the selected entry
df.at[index, "keywords"].append(new_keyword)

# Save the entire updated DataFrame to CSV
df.to_csv(f"{roll_number}_faq_data.csv", index=False)
print("Updated DataFrame:")
print(df)

print(f"\nFile saved as {roll_number}_faq_data.csv")

Updated DataFrame:
                                            question  \
0        How can I reset my student portal password?   
1              How can I pay my tuition fees online?   
2              Where can I check my class timetable?   
3             How can I contact an academic advisor?   
4  Where can I find information about college hol...   
5  How can I check whether my tuition fee payment...   

                                              answer  \
0  Use the 'Forgot Password' link on the student ...   
1  Go to the billing section in the student porta...   
2  Your class timetable is available under the Ac...   
3  You can contact an academic advisor through th...   
4  You can check the academic calendar in the stu...   
5  You can verify your tuition fee payment status...   

                                    keywords category  
0  [password, login, security,student, name]  account  
1                   [fees, payment, receipt]  billing  
2             [timetable, c

In [19]:
category_counts = df.groupby("category").size()

print(category_counts)

category
account    1
billing    2
general    3
dtype: int64


In [20]:
def score_hypothesis(query):
    """
    Score FAQ rows by keyword overlap.
    If several rows tie for the highest confidence score,
    print every tied highest-score match instead of silently
    choosing one. If there is no tie, print the single top match.
    """
    query_words = set(query.lower().split())
    results = []

    for _, row in df.iterrows():
        keywords = set(word.lower() for word in row["keywords"])
        matches = query_words.intersection(keywords)

        if len(keywords) > 0:
            confidence = len(matches) / len(keywords)
        else:
            confidence = 0

        if confidence > 0:
            results.append({
                "question": row["question"],
                "answer": row["answer"],
                "category": row["category"],
                "matched_keywords": sorted(matches),
                "confidence": confidence
            })

    
    results.sort(key=lambda x: x["confidence"], reverse=True)

    if not results:
        print(f"No matches for query: {query}")
        return []

    
    best_score = results[0]["confidence"]

    tied_best = [r for r in results if r["confidence"] == best_score]

    if len(tied_best) > 1:
        print(f"\nQuery: {query}")
        print(f"Top score tie detected at confidence = {best_score:.3f}")
        print("Printing all equally good matches:")
        for item in tied_best:
            print(item)
        return tied_best

    print(f"\nQuery: {query}")
    print(f"No top-score tie detected. Unique highest score = {best_score:.3f}")
    print("Printing the top result only:")
    print(results[0])
    return [results[0]]

print("=== Tie demonstration ===")
tie_query = "payment receipt"
score_hypothesis(tie_query)

print("\n=== Non-tie demonstration ===")
non_tie_query = "Where can I check my class timetable?"
score_hypothesis(non_tie_query)

=== Tie demonstration ===

Query: payment receipt
Top score tie detected at confidence = 0.667
Printing all equally good matches:
{'question': 'How can I pay my tuition fees online?', 'answer': 'Go to the billing section in the student portal, select your term, and complete the payment using the secure payment gateway.', 'category': 'billing', 'matched_keywords': ['payment', 'receipt'], 'confidence': 0.6666666666666666}
{'question': 'How can I check whether my tuition fee payment was successful?', 'answer': 'You can verify your tuition fee payment status in the billing section of the student portal and view the payment receipt.', 'category': 'billing', 'matched_keywords': ['payment', 'receipt'], 'confidence': 0.6666666666666666}

=== Non-tie demonstration ===
No matches for query: Where can I check my class timetable?


[]